In [63]:
from pathlib import Path
import re
import pandas as pd

CORPUS_DIR = Path(r"Corpus/")   # output folder of beloit_ingest.py
CHUNK_WORDS = 750                                                # max words per chunk
HEADER_RULE = "-" * 60                                           # line that ends each file's header

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

In [64]:
def parse_document(path):
    """Return (metadata dict, body text) for one corpus file."""
    raw = path.read_text(encoding="utf-8").replace("\r\n", "\n")   # files may have Windows line endings
    if HEADER_RULE not in raw:
        raise ValueError("no header rule (a line of 60 dashes) found")
    head, body = raw.split(HEADER_RULE, 1)

    meta = {}
    for line in head.splitlines():
        if ":" in line:
            key, value = line.split(":", 1)
            meta[key.strip().lower()] = value.strip()

    body = body.strip()
    if not body:
        raise ValueError("body text is empty")
    return meta, body


def load_corpus(corpus_dir):
    if not corpus_dir.exists():
        raise FileNotFoundError(f"corpus folder not found: {corpus_dir}")
    docs, problems = [], []
    for path in sorted(corpus_dir.glob("*.txt")):
        try:
            meta, body = parse_document(path)
        except Exception as e:
            problems.append((path.name, str(e)))
            continue
        docs.append({
            "doc_id": meta.get("id", path.stem),
            "title": meta.get("title", ""),
            "source": meta.get("source", ""),
            "kind": meta.get("kind", ""),
            "file": path.name,
            "text": body,
            "words": len(body.split()),
        })
    return docs, problems


docs, problems = load_corpus(CORPUS_DIR)
print(f"Loaded {len(docs)} documents from {CORPUS_DIR}")
if problems:
    print(f"\nSkipped {len(problems)} file(s):")
    for name, why in problems:
        print(f"  {name}: {why}")

Loaded 142 documents from Corpus


In [65]:
def chunk_text(text, max_words=CHUNK_WORDS):
    """
    Split text into consecutive pieces of at most `max_words` words.
    Pieces are cut from the original string, so line breaks and paragraph breaks
    inside a chunk are kept exactly as they were in the file.
    """
    words = list(re.finditer(r"\S+", text))
    if len(words) <= max_words:
        return [text.strip()]
    pieces = []
    for start in range(0, len(words), max_words):
        group = words[start:start + max_words]
        pieces.append(text[group[0].start():group[-1].end()])
    return pieces


def chunk_documents(docs, max_words=CHUNK_WORDS):
    rows = []
    for d in docs:
        pieces = chunk_text(d["text"], max_words)
        for i, piece in enumerate(pieces, start=1):
            rows.append({
                "chunk_id": re.sub(r"\s+", "_", f"{d['doc_id']}__{i:02d}"),
                "doc_id": d["doc_id"],
                "title": d["title"],
                "source": d["source"],
                "kind": d["kind"],          # <-- added: carries the page range
                "chunk_no": i,
                "n_chunks": len(pieces),
                "words": len(piece.split()),
                "text": piece,
            })
    return pd.DataFrame(rows)


chunks = chunk_documents(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks")

142 documents -> 369 chunks


In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import Normalizer
import numpy as np
import joblib

INDEX_FILE = CORPUS_DIR.parent / "index" / "tfidf_index.joblib"

embedder = Pipeline([
    ("features", FeatureUnion([
        ("words", TfidfVectorizer(ngram_range=(1, 2), stop_words="english",
                                  sublinear_tf=True, strip_accents="unicode")),
        ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                  sublinear_tf=True, min_df=2, strip_accents="unicode")),
    ])),
    ("unit_length", Normalizer()),
])

matrix = embedder.fit_transform(chunks["text"])      # one row per chunk, same order as `chunks`
print(f"embedded {matrix.shape[0]} chunks into {matrix.shape[1]:,} features "
      f"({matrix.nnz / matrix.shape[0]:,.0f} non-zero values per chunk)")

embedded 369 chunks into 116,548 features (3,684 non-zero values per chunk)


In [67]:
INDEX_FILE.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"embedder": embedder, "matrix": matrix, "chunks": chunks}, INDEX_FILE)
print(f"saved index to {INDEX_FILE}  ({INDEX_FILE.stat().st_size / 1e6:.1f} MB)")

saved index to index/tfidf_index.joblib  (20.7 MB)


In [69]:
N_CHUNKS = 3               # chunks kept from the first search
N_PARAGRAPHS = 3           # paragraphs returned
MIN_PARAGRAPH_WORDS = 20   # shorter pieces are joined to the paragraph that follows


def split_paragraphs(text, min_words=MIN_PARAGRAPH_WORDS):
    parts = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    merged, carry = [], ""
    for p in parts:
        p = f"{carry}\n{p}" if carry else p
        if len(p.split()) < min_words:
            carry = p
        else:
            merged.append(p)
            carry = ""
    if carry:                                   # short piece at the very end
        if merged:
            merged[-1] += "\n" + carry
        else:
            merged.append(carry)
    return merged


def search(question, n_chunks=N_CHUNKS, n_paragraphs=N_PARAGRAPHS):
    q = embedder.transform([question])
    chunk_scores = (matrix @ q.T).toarray().ravel()
    best = np.argsort(chunk_scores)[::-1][:n_chunks]

    rows = []
    for i in best:
        c = chunks.iloc[i]
        for p in split_paragraphs(c["text"]):
            rows.append({
                "chunk_id": c["chunk_id"], 
                "title": c["title"],
                "kind": c.get("kind", ""),          
                "source": c["doc_id"],
                "paragraph": p
            })
            
    scores = (embedder.transform([r["paragraph"] for r in rows]) @ q.T).toarray().ravel()
    out = pd.DataFrame(rows).assign(score=scores)
    return out.sort_values("score", ascending=False).head(n_paragraphs).reset_index(drop=True)


def show(question, **kw):
    print(f"QUESTION: {question}\n")
    for rank, r in search(question, **kw).iterrows():
        print(f"[{rank + 1}] score {r['score']:.3f}  |  {r['title']}  |  {r['kind']}  |  {r['chunk_id']}")
        print(r["paragraph"], "\n")

In [81]:
from google import genai
from google.genai import types

client = genai.Client(
    vertexai=True,
    project="dsda310",
    location="global"
)

In [82]:
schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "answer_found": types.Schema(
            type=types.Type.BOOLEAN,
            description="True if context contains the answer, False if answer is missing."
        ),
        "answer": types.Schema(
            type=types.Type.STRING,
            description="The answer to the user's question, or a refusal if answer is missing."
        ),
        "supporting_quote": types.Schema(
            type=types.Type.STRING,
            description="The exact quote from the context that contains the answer. Empty if answer is missing."
        ),
        "source_doc": types.Schema(
            type=types.Type.STRING,
            description="The source filename that the quote came from. Empty if answer is missing."
        ),
    },
    required=["answer_found", "answer", "supporting_quote", "source_doc"]
)

In [83]:
def query(question):
    retrieved = search(question, n_paragraphs=3)
    
    context = ""
    for _, row in retrieved.iterrows():
        context += f"\nSource: {row['source']}\n{row['paragraph'].strip()}\n"

    prompt = f"""
    # You are answering a question from a Beloit College student. Use only the provided context to answer the question. If the context does not have the answer to the question, set answer_found to False, and refuse to answer.

    Context:
    {context}

    Question:
    {question}
    """

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=schema,
                temperature=0.0
            )
        )
        return response.text
    except Exception as e:
        print(f"API Error: {str(e)}")
        return "{}"

    return "{}"

In [84]:
eval_set = [
    {
        "question": "Are weapons are prohibited on campus?",
        "source": "hb_section3_o"
    },
    {
        "question": "Can I drink in any academic buildings?",
        "source": "hb_section3_b1"
    },
    {
        "question": "Can I use spraychalk on beloit campus/sidewalks?",
        "source": "hb_section3_d3"
    },
    {
        "question": "What is the lock out policy on campus?",
        "source": "hb_section3_g2"
    },
    {
        "question": "Who are the computer science major/minor faculty ?",
        "source": "cc_chapter2_computer_science"
    },
    {
        "question": "What is the academic honesty policy on campus?",
        "source": "hb_section3_a1"
    },
    {
        "question": "Is Introduction to Object-Oriented Programming a prerequisite for Web Design?",
        "source": "cc_chapter2_computer_science"
    },
    {
        "question": "When does summer term begin in 2027?",
        "source": "cc_acedemic_calanders"
    },
    {
        "question": "What are the four princples of Restorative Justice?",
        "source": "hb_section2_a"
    },
    {
        "question": "Can the college refuse transcripts?",
        "source": "hb_section3_a4"
    },
    {
        "question": "Can a dog be an ESA?",
        "source": "hb_section3_f1"
    },
    {
        "question": "How to get Departmental Honors in Data Science & Data Analytics?",
        "source": "cc_chapter2_data_science_&_analytics"
    },
    {
        "question": "Who is the Community Connections Faculty?",
        "source": "cc_chapter3_community_connections"
    },
    {
        "question": "What are the requirements for Japanese Minor?",
        "source": "cc_chapter2_modern_languages_&_literatures"
    },
    {
        "question": "How many units is a Greek and Latin Studies Major?",
        "source": "cc_chapter2_glam"
    },
    {
        "question": "What are all the Business and Economic majors?",
        "source": "cc_chapter2_index"
    },
    {
        "question": "Does Beloit College have an Engineering Degree?",
        "source": "cc_chapter2_engineering"
    },
    {
        "question": "What is the appeal process for summer and vacation periods?",
        "source": "hb_section4"
    },
    {
        "question": "What degrees are offered at Beloit College?",
        "source": "cc_chapter1_skill_requirements"
    },
    {
        "question": "Who are the Critical Identity Studies Faculty?",
        "source": "cc_chapter2_critical_identity_studies"
    },
    {
        "question": "What are the responsibilities of College security staff?",
        "source": "hb_section3_k_intro"
    },
    {
        "question": "What are the two primary functions of the mail center?",
        "source": "hb_section3_h"
    },
    {
        "question": "Are space heaters in residence spaces?",
        "source": "hb_section3_g2"
    },
    {
        "question": "Are Air conditioning units allowed on campus?",
        "source": "hb_section3_g2"
    },
    {
        "question": "What is the Summer Tuition for Special Projects and Internships?",
        "source": "cc_chapter5_tuition"
    },
    {
        "question": "Who are the Morse Library Staff?",
        "source": "cc_chapter4_morse_library"
    },
    {
        "question": "Best places to live off campus?",
        "source": "none"
    },
    {
        "question": "Does the resident building Chapin has a good vibe?",
        "source": "none"
    },
    {
        "question": "Does the Residential Life Director answer emails from parents?",
        "source": "none"
    }, 
    {
        "question": "Can I use a macbook in my Data Anayltics class?",
        "source": "none"
    }
]

In [85]:
import json, time
def norm(s):
    return " ".join((s or "").lower().split())

def strip_ext(s):
    """Helper to remove .txt extension for safe matching."""
    return s.replace(".txt", "") if s else ""

rows = []
for i, item in enumerate(eval_set):
    retrieved = search(item["question"], n_paragraphs=3)
    retrieved_sources = retrieved["source"].tolist()

    raw_response = query(item["question"])
    result = json.loads(raw_response) if raw_response else {}

    ans_found = result.get("answer_found", False)
    quote = result.get("supporting_quote", "")
    cited_doc = result.get("source_doc", "")

    clean_cited_doc = strip_ext(cited_doc)
    clean_retrieved_sources_series = retrieved["source"].apply(strip_ext)
    
    cited_paragraphs = retrieved[clean_retrieved_sources_series == clean_cited_doc]["paragraph"].tolist()
    combined_cited_text = norm(" ".join(cited_paragraphs))

    returned_quote = int(bool(quote and quote.strip()))
    quote_found = int(norm(quote) in combined_cited_text) if returned_quote else 0

    expected_doc = item.get("source")
    
    clean_expected_doc = strip_ext(expected_doc)
    clean_retrieved_sources = [strip_ext(s) for s in retrieved_sources]

    rows.append({
        "q":              item["question"],
        # Check that it exists and is not the string "none"
        "answerable":     bool(expected_doc and expected_doc.lower() != "none"),
        "hit":            int(clean_expected_doc in clean_retrieved_sources) if expected_doc and expected_doc.lower() != "none" else None,
        "returned_quote": returned_quote,
        "quote_found":    quote_found,
        "correct":        None,
        "refused":        int(not ans_found),
    })
    
    print(f"Question {i+1}: {item["question"]}\nAnswer: {result.get('answer', 'No answer provided')}\n")
    for rank, r in retrieved.iterrows():
        print(f"[{rank + 1}] score {r['score']:.3f}  |  {r['title']}  |  {r['kind']}  |  {r['chunk_id']}")
        print(r["paragraph"], "\n")

Question 1: Are weapons are prohibited on campus?
Answer: Yes, weapons are strictly prohibited on College-owned or controlled property, regardless of whether the weapon is legal or if an individual holds a permit or license.

[1] score 0.383  |  Student Handbook Policies and Regulations G2  |  pdf pages 68-78  |  hb_section3_g2__07
Weapons and Explosives: The use and/or storage of any weapons, firearms, ammunition, or
explosives of any kind in the residence halls are strictly prohibited. Laboratory chemicals and fireworks
also are not permitted in the residence halls. (See: Weapons.) 

[2] score 0.235  |  Student Handbook Policies and Regulations O  |  pdf pages 114  |  hb_section3_o__01
Note regarding graffiti: “Graffiti” is defined as any unauthorized writing, drawing or marking
created on any campus structure, public or private. Regardless of whether or not graffiti is written
in permanent materials, it will be considered a form of vandalism. If graffiti is found to contain
messagin

In [80]:
df = pd.DataFrame(rows)

correctness = [
    1, 
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
]

ans   = df[df['answerable'] == True]
unans = df[df['answerable'] == False]
df['correct'] = correctness

hit_rate        = f"{int(ans['hit'].sum())} of {len(ans)}"
quote_fidelity  = f"{int(df['quote_found'].sum())} of {int(df['returned_quote'].sum())}"
correct_answers = f"{int(df['correct'].sum())} of {len(df)}"

if len(unans) > 0:
    correct_refusal = f"{int(unans['refused'].sum())} of {len(unans)}"
else:
    correct_refusal = "0 of 0"

wrong_refusal   = f"{int(ans['refused'].sum())} of {len(ans)}"

print(f"hit rate:         {hit_rate}")
print(f"quote fidelity:   {quote_fidelity}")
print(f"correctness:      {correct_answers}")
print(f"correct refusals: {correct_refusal}")
print(f"wrong refusals:   {wrong_refusal}")

hit rate:         23 of 26
quote fidelity:   19 of 19
correctness:      23 of 30
correct refusals: 4 of 4
wrong refusals:   7 of 26
